# Cyclistic Bike Share — SQL Prototype

Input: **`trips_prototype.csv`** (~148k rows). Output: **`cyclistic.db`** beside this notebook.


SQLite is not representative of SQL performance in general - it has no parallel execution, no vectorised query engine, and its to_sql ingest is not optimised for bulk loading. It is used here specifically because its single-threaded architecture makes the bottlenecks visible and its behaviour predictable, providing a clean sequential baseline against which MapReduce parallelism can be measured.

## Setup & Hardware Specs
Record CPU, RAM, and disk so runtimes can be interpreted in context.

In [1]:
import json
import os
import platform
import shutil
import sqlite3
import sys
import time
from pathlib import Path


import pandas as pd
import psutil
from sqlalchemy import create_engine, text

notebook_start = time.perf_counter()
timings: dict = {}

print("===== CPU =====")
print(f"Processor      : {platform.processor() or 'n/a'}")
print(f"Physical cores : {psutil.cpu_count(logical=False)}")
print(f"Logical CPUs   : {psutil.cpu_count(logical=True)}")

print("\n===== RAM =====")
vm = psutil.virtual_memory()
print(f"Total     : {vm.total / (1024**3):.2f} GB")
print(f"Available : {vm.available / (1024**3):.2f} GB")

print("\n===== Disk (cwd) =====")
usage = shutil.disk_usage(".")
print(f"Total : {usage.total / (1024**3):.2f} GB")
print(f"Free  : {usage.free  / (1024**3):.2f} GB")

print("\n===== Python =====")
print(sys.version)

print("\n===== SQLite =====")
print(f"sqlite3 module: {sqlite3.sqlite_version}")

===== CPU =====
Processor      : Intel64 Family 6 Model 126 Stepping 5, GenuineIntel
Physical cores : 2
Logical CPUs   : 4

===== RAM =====
Total     : 7.77 GB
Available : 1.12 GB

===== Disk (cwd) =====
Total : 236.86 GB
Free  : 22.02 GB

===== Python =====
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

===== SQLite =====
sqlite3 module: 3.49.1


## Load CSV → SQLite
Read `trips_prototype.csv` with pandas and ingest into SQLite via `to_sql`.  
Timed as a single block — the SQL equivalent of the pandas load + preprocess step.

In [2]:
def _notebook_dir() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()

NOTEBOOK_DIR = _notebook_dir()
CSV_PATH = NOTEBOOK_DIR / "trips_prototype.csv"
if not CSV_PATH.is_file():
    alt = Path.cwd() / "Prototype" / "trips_prototype.csv"
    if alt.is_file():
        CSV_PATH = alt
        NOTEBOOK_DIR = CSV_PATH.parent
    else:
        raise FileNotFoundError(
            f"trips_prototype.csv not found. Tried: {NOTEBOOK_DIR / 'trips_prototype.csv'} and {alt}"
        )

DB_PATH = NOTEBOOK_DIR / "cyclistic.db"
print(f"Using CSV : {CSV_PATH.resolve()}")
print(f"SQLite DB : {DB_PATH.resolve()}")

if DB_PATH.exists():
    DB_PATH.unlink()

_url = "sqlite:///" + DB_PATH.resolve().as_posix()
engine = create_engine(_url, pool_pre_ping=True)

DDL = """
CREATE TABLE trips (
    ride_id            TEXT PRIMARY KEY,
    rideable_type      TEXT,
    started_at         TEXT,
    ended_at           TEXT,
    start_station_name TEXT,
    start_station_id   TEXT,
    end_station_name   TEXT,
    end_station_id     TEXT,
    start_lat          REAL,
    start_lng          REAL,
    end_lat            REAL,
    end_lng            REAL,
    member_casual      TEXT
);
"""

# Duration helper (SQLite has no TIMESTAMPDIFF; MySQL uses TIMESTAMPDIFF(MINUTE, started_at, ended_at))
DUR_MIN = "(julianday(ended_at) - julianday(started_at)) * 24 * 60"

t0 = time.perf_counter()

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS trips"))
    conn.execute(text(DDL))

df_raw = pd.read_csv(CSV_PATH, dtype={"ride_id": str})
df_raw.to_sql("trips", engine, if_exists="append", index=False, chunksize=8000)

load_s = time.perf_counter() - t0
timings["load_ingest"] = round(load_s, 4)

print(f"Rows ingested : {len(df_raw):,}")
print(f"Elapsed       : {load_s:.4f}s")

def run_sql(sql: str, *, label: str | None = None):
    """Execute SQL, return (DataFrame, elapsed_seconds)."""
    if label:
        print(label)
    t0 = time.perf_counter()
    result = pd.read_sql(text(sql), engine)
    elapsed = time.perf_counter() - t0
    print(f"Elapsed: {elapsed:.4f}s")
    return result, elapsed

Using CSV : C:\Users\imonl\BigData\Cyclistic_Data\Prototype\trips_prototype.csv
SQLite DB : C:\Users\imonl\BigData\Cyclistic_Data\Prototype\cyclistic.db
Rows ingested : 148,045
Elapsed       : 4.5747s


## Row Count & Data Quality
Full-table scan: total rows, missing station names, missing end coordinates, invalid durations.  
Equivalent to the pandas `.isna()` + boolean mask quality checks.

In [3]:
df_quality, t_quality = run_sql(
    f"""
SELECT
    COUNT(*)                                                                   AS total_rows,
    SUM(CASE WHEN start_station_name IS NULL OR start_station_name = '' THEN 1 ELSE 0 END) AS missing_start_station,
    SUM(CASE WHEN end_station_name   IS NULL OR end_station_name   = '' THEN 1 ELSE 0 END) AS missing_end_station,
    SUM(CASE WHEN end_lat IS NULL OR end_lng IS NULL                    THEN 1 ELSE 0 END) AS missing_end_coords,
    SUM(CASE WHEN ({DUR_MIN}) <= 0                                      THEN 1 ELSE 0 END) AS invalid_durations
FROM trips;
    """,
    label="===== Row count & data quality =====",
)
timings["row_count_quality"] = round(t_quality, 4)
display(df_quality)

===== Row count & data quality =====
Elapsed: 0.1028s


,total_rows,missing_start_station,missing_end_station,missing_end_coords,invalid_durations
0,148045,16102,17453,165,106


## Rides by Member Type & Bike Type
Two simple GROUP BY aggregations. Low cardinality — fast everywhere.  
Timed individually, matching the pandas notebook structure.

In [4]:
df_member, t_member = run_sql(
    """
SELECT member_casual, COUNT(*) AS total_rides
FROM trips
GROUP BY member_casual
ORDER BY total_rides DESC;
    """,
    label="===== Rides by member type =====",
)
timings["rides_by_member_type"] = round(t_member, 4)
display(df_member)

df_bike, t_bike = run_sql(
    """
SELECT rideable_type, COUNT(*) AS total_rides
FROM trips
GROUP BY rideable_type
ORDER BY total_rides DESC;
    """,
    label="\n===== Rides by bike type =====",
)
timings["rides_by_bike_type"] = round(t_bike, 4)
display(df_bike)

===== Rides by member type =====
Elapsed: 0.0941s


,member_casual,total_rides
0,member,86000
1,casual,62045



===== Rides by bike type =====
Elapsed: 0.0905s


,rideable_type,total_rides
0,classic_bike,59391
1,electric_bike,54176
2,docked_bike,34478


## Average Ride Duration by Member Type
Inline duration arithmetic using `julianday()` — equivalent to the pandas `dt.total_seconds()` aggregation.

In [5]:
df_duration, t_duration = run_sql(
    f"""
SELECT
    member_casual,
    ROUND(AVG({DUR_MIN}), 2) AS avg_duration_minutes,
    ROUND(MIN({DUR_MIN}), 2) AS min_duration_minutes,
    ROUND(MAX({DUR_MIN}), 2) AS max_duration_minutes
FROM trips
WHERE ({DUR_MIN}) > 0
GROUP BY member_casual;
    """,
    label="===== Avg ride duration by member type =====",
)
timings["avg_duration_by_member"] = round(t_duration, 4)
display(df_duration)

===== Avg ride duration by member type =====
Elapsed: 0.2753s


,member_casual,avg_duration_minutes,min_duration_minutes,max_duration_minutes
0,casual,32.32,0.02,20679.02
1,member,13.70,0.02,1499.93


## Popular Routes *(MapReduce Candidate 1)*
GROUP BY on the composite key `(start_station_name, end_station_name)`.  
At prototype scale this produces ~10k–50k unique key pairs; at 14.8M rows it exceeds 500k.  
**Why?** the key space grows quadratically with stations, 
making SQLite's single-threaded B-tree aggregation the bottleneck. 
Spark's distributed shuffle + hash aggregation eliminates that bottleneck by partitioning keys across workers.

In [6]:
df_routes, t_routes = run_sql(
    """
SELECT
    start_station_name,
    end_station_name,
    COUNT(*) AS route_count
FROM trips
WHERE start_station_name IS NOT NULL AND start_station_name != ''
  AND end_station_name   IS NOT NULL AND end_station_name   != ''
GROUP BY start_station_name, end_station_name
ORDER BY route_count DESC
LIMIT 10;
    """,
    label="===== Popular routes (MapReduce Candidate 1) =====",
)
timings["popular_routes_mapreduce_1"] = round(t_routes, 4)
display(df_routes)

===== Popular routes (MapReduce Candidate 1) =====
Elapsed: 0.2000s


,start_station_name,end_station_name,route_count
0,Streeter Dr & Grand Ave,Streeter Dr & Grand Ave,322
1,Millennium Park,Millennium Park,163
2,Michigan Ave & Oak St,Michigan Ave & Oak St,161
3,Ellis Ave & 60th St,University Ave & 57th St,136
4,Ellis Ave & 60th St,Ellis Ave & 55th St,130
5,Indiana Ave & Roosevelt Rd,Indiana Ave & Roosevelt Rd,128
6,Ellis Ave & 55th St,Ellis Ave & 60th St,123
7,DuSable Lake Shore Dr & Monroe St,DuSable Lake Shore Dr & Monroe St,120
8,University Ave & 57th St,Ellis Ave & 60th St,119
9,Lake Shore Dr & Monroe St,Lake Shore Dr & Monroe St,116


## Busiest Stations — Combined Traffic *(MapReduce Candidate 2)*
UNION ALL of departures and arrivals per station, then outer GROUP BY + SUM.  
**Why?** two full-table scans feed a single reduce step — 
the canonical MapReduce word-count pattern with stations as keys. 
Spark can execute both scans in parallel on separate data partitions; 
SQLite and pandas execute them sequentially.

In [7]:
df_stations, t_stations = run_sql(
    """
SELECT
    station_name,
    SUM(activity_count) AS total_activity
FROM (
    SELECT start_station_name AS station_name, COUNT(*) AS activity_count
    FROM trips
    WHERE start_station_name IS NOT NULL AND start_station_name != ''
    GROUP BY start_station_name

    UNION ALL

    SELECT end_station_name AS station_name, COUNT(*) AS activity_count
    FROM trips
    WHERE end_station_name IS NOT NULL AND end_station_name != ''
    GROUP BY end_station_name
) AS combined_traffic
GROUP BY station_name
ORDER BY total_activity DESC
LIMIT 10;
    """,
    label="===== Busiest stations — combined traffic (MapReduce Candidate 2) =====",
)
timings["busiest_stations_mapreduce_2"] = round(t_stations, 4)
display(df_stations)

===== Busiest stations — combined traffic (MapReduce Candidate 2) =====
Elapsed: 0.2359s


,station_name,total_activity
0,Streeter Dr & Grand Ave,3868
1,Michigan Ave & Oak St,2265
2,Clark St & Elm St,2190
3,Wells St & Concord Ln,2160
4,Millennium Park,2109
5,Theater on the Lake,1981
6,Wells St & Elm St,1831
7,Clark St & Lincoln Ave,1741
8,Indiana Ave & Roosevelt Rd,1716
9,Broadway & Barry Ave,1706


---
## Summary: Performance Baseline
All per-step timings, total wall time, and memory usage. Written to `metrics.json`.

In [8]:
notebook_end = time.perf_counter()
total_s = notebook_end - notebook_start

process = psutil.Process(os.getpid())
memory_mb = process.memory_info().rss / (1024 * 1024)

print("=" * 50)
print("SQL PROTOTYPE — TIMING SUMMARY")
print("=" * 50)
labels = {
    "load_ingest"                 : "Load CSV → SQLite",
    "row_count_quality"           : "Row count & data quality",
    "rides_by_member_type"        : "Rides by member type",
    "rides_by_bike_type"          : "Rides by bike type",
    "avg_duration_by_member"      : "Avg duration by member type",
    "popular_routes_mapreduce_1"  : "Popular routes (MR Candidate 1)",
    "busiest_stations_mapreduce_2": "Busiest stations (MR Candidate 2)",
}
for key, label in labels.items():
    val = timings.get(key, 0)
    print(f"  {label:<38} {val:.4f}s")

print("-" * 50)
print(f"  {'Total wall time':<38} {total_s:.4f}s")
print(f"  {'Memory (RSS)':<38} {memory_mb:.2f} MB")
print("=" * 50)

# Write to shared metrics.json
results_dir = Path(NOTEBOOK_DIR) / "results"
if not results_dir.exists():
    results_dir = Path(NOTEBOOK_DIR).parent / "results"
results_dir.mkdir(parents=True, exist_ok=True)

metrics_path = results_dir / "metrics.json"
existing = {}
if metrics_path.exists():
    try:
        existing = json.loads(metrics_path.read_text())
    except json.JSONDecodeError:
        pass

existing["sql_prototype"] = {
    "steps"                        : timings,
    "total_execution_time_seconds" : round(total_s, 2),
    "memory_usage_mb"              : round(memory_mb, 2),
}
metrics_path.write_text(json.dumps(existing, indent=4))
print(f"\nMetrics written to: {metrics_path.resolve()}")

SQL PROTOTYPE — TIMING SUMMARY
  Load CSV → SQLite                      4.5747s
  Row count & data quality               0.1028s
  Rides by member type                   0.0941s
  Rides by bike type                     0.0905s
  Avg duration by member type            0.2753s
  Popular routes (MR Candidate 1)        0.2000s
  Busiest stations (MR Candidate 2)      0.2359s
--------------------------------------------------
  Total wall time                        5.7863s
  Memory (RSS)                           165.06 MB

Metrics written to: C:\Users\imonl\BigData\Cyclistic_Data\results\metrics.json
